In [1]:
import os
import sys
import pickle
import numpy as np

# sys.path.append('/n/home04/aboesky/berger/Weird_Galaxies')
sys.path.append('/Users/adamboesky/Research/ay98/Weird_Galaxies')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
import astropy.units as u

from astropy.io import ascii
from astropy.table import vstack
from astropy.coordinates import SkyCoord

from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, precision_recall_curve

plt.rc('text', usetex=True)
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = 'cmr10'  # Computer Modern Roman
mpl.rcParams['font.size'] = 12  # Adjust the font size as needed

# Load data

In [22]:
# BTS
bts_hosts = ascii.read('/Users/adamboesky/Research/ay98/clean_data/bts_hosts.ecsv')

# YSE
yse_hosts = ascii.read('/Users/adamboesky/Research/ay98/clean_data/yse_hosts.ecsv')

# Combine
sn_tab = vstack((bts_hosts, yse_hosts))
sn_tab

objID_3pi,raStack_3pi,decStack_3pi,primaryDetection_3pi,gKronMag_3pi,rKronMag_3pi,iKronMag_3pi,zKronMag_3pi,yKronMag_3pi,gKronMagErr_3pi,rKronMagErr_3pi,iKronMagErr_3pi,zKronMagErr_3pi,yKronMagErr_3pi,ps_score_3pi,SN_ra,SN_dec,sn_class,sn_redshift,ZTFID
str18,str12,str12,str1,str7,str7,str7,str7,str7,str12,str12,str12,str12,str12,str10,float64,float64,str14,float64,str12
126681757766820931,175.77662082,15.56701789,1,16.0256,15.2881,14.9533,14.7114,14.5005,0.002978,0.001497,0.001272,0.001197,0.002235,0.1110298,175.77654166666665,15.56713888888889,--,--,--
148231465149294617,146.51493972,33.52836114,1,18.2387,17.9746,17.875,17.8532,17.7821,0.004157,0.004871,0.004281,0.014407,0.019814,0.021125,146.51479166666667,33.52825,SN II,0.038,ZTF18aacemcn
153731349546771375,134.95471695,38.10897029,1,18.7478,19.0585,18.6466,18.8301,18.6221,0.006392,0.009475,0.004919,0.017235,0.038667,0.01741071,134.95466666666664,38.10905555555556,SN II,0.07247,ZTF18aacnlxz
169101206490177628,120.64903126,50.9224606,1,16.5325,15.6522,15.2551,15.0064,14.7582,0.002335,0.002128,0.000867,0.001623,0.003512,0.07590476,120.64895833333333,50.92252777777777,SN Ia,0.05295,ZTF18aadlaxo
159661026075384557,102.60753619,43.05326583,1,nan,14.2309,13.166,12.9816,12.8047,nan,0.000836,0.000408,0.001003,0.001687,0.1546488,102.60754166666663,43.05322222222222,SN IIn,0.01885,ZTF18aadmssd
146001295340652041,129.53408241,31.66789412,1,17.2694,16.9697,16.8672,16.8232,16.7636,0.004272,0.003413,0.005182,0.005542,0.014564,0.00466667,129.53395833333332,31.667916666666667,SN II,0.03057,ZTF18aadsuxd
160090985073071380,98.50726462,43.40895049,1,17.7127,16.8614,16.3023,16.0124,15.8248,0.005224,0.003569,0.001682,0.003451,0.005626,0.02679167,98.50770833333333,43.40902777777777,SN Ia,0.064,ZTF18aadxnul
159981068237030321,106.82370726,43.31640285,1,16.9512,16.1689,15.6894,15.5194,15.3639,0.003268,0.002049,0.001297,0.002374,0.005479,0.02479167,106.823625,43.31641666666666,SN Ia,0.05,ZTF18aadzfso
135811154938631883,115.49382081,23.17596697,1,15.7987,nan,nan,nan,nan,0.001466,nan,nan,nan,nan,0.2847917,115.49429166666664,23.176472222222223,SN Ia,0.043,ZTF18aaeqjmc


In [23]:
# Drop rows without SN classification
sn_tab = sn_tab[~sn_tab['sn_class'].mask]
sn_tab

# Add the angular separation between SN and host galaxy in arcseconds
sn_coords = SkyCoord(ra=sn_tab['SN_ra'], dec=sn_tab['SN_dec'], unit='deg')
host_coords = SkyCoord(ra=sn_tab['raStack_3pi'], dec=sn_tab['decStack_3pi'], unit='deg')
sn_tab['sep_arcsec'] = sn_coords.separation(host_coords).arcsec

sn_df = sn_tab.to_pandas()

In [24]:
classification_groups = [['ia'], ['ib', 'ic'], ['slsn'], ['iin'], ['ii', 'iip', 'ii p', 'iil', 'ii l']]

def add_sn_class_cols(df: pd.DataFrame) -> pd.DataFrame:
    # Ia or other
    ia_mask = np.char.find(np.char.lower(np.array([str(item) for item in df['sn_class']], dtype=str)), 'ia') != -1
    df['sn_class_binary'] = ia_mask.astype(int)

    # Ia, II, or other
    # Conditions to check for 'ii' or 'ia' in a case-insensitive manner
    condition_ii = np.vectorize(lambda s: 'ii' in s.lower())
    condition_ia = np.vectorize(lambda s: 'ia' in s.lower())

    # Apply conditions and choices to the string array
    df['sn_class_tertiary'] = np.where(condition_ia(df['sn_class']), 0, np.where(condition_ii(df['sn_class']), 1, 2))

    # Your classification groups
    def add_category_id_column(my_df, classification_groups):
        # Convert classification groups to lowercase and assign unique integers to each category
        lowercase_groups = {cls.lower(): i for i, group in enumerate(classification_groups) for cls in group}

        # Function to get the category id for a supernova class
        def get_category_id(sn_class_list):
            for sn_class in sn_class_list:
                sn_class_lower = sn_class.lower()
                for group, group_id in lowercase_groups.items():
                    if group in sn_class_lower:
                        return group_id
            return np.max(list(lowercase_groups.values())) + 1  # Return a default id if no category is found

        # Add 'category_id' column to the DataFrame
        my_df['sn_class_cat'] = my_df['sn_class'].apply(lambda x: get_category_id(x.split(',')))

        return my_df

    # Apply the function to the DataFrame
    df = add_category_id_column(df, classification_groups)
    for idx, snt in enumerate(('ia', 'ib/c', 'slsn', 'iin', 'ii (p/l)', 'other')):
        print(f'The data are {np.round(len(df[df["sn_class_cat"] == idx]) / len(df) * 100, 2)}% type {snt} with n = {len(df[df["sn_class_cat"] == idx])}')

    return df

In [25]:
sn_df = add_sn_class_cols(sn_df)
sn_df

The data are 75.15% type ia with n = 3683
The data are 7.37% type ib/c with n = 361
The data are 0.65% type slsn with n = 32
The data are 2.37% type iin with n = 116
The data are 13.94% type ii (p/l) with n = 683
The data are 0.53% type other with n = 26


,objID_3pi,raStack_3pi,decStack_3pi,primaryDetection_3pi,gKronMag_3pi,rKronMag_3pi,iKronMag_3pi,zKronMag_3pi,yKronMag_3pi,gKronMagErr_3pi,...,ps_score_3pi,SN_ra,SN_dec,sn_class,sn_redshift,ZTFID,sep_arcsec,sn_class_binary,sn_class_tertiary,sn_class_cat
0,148231465149294617,146.51493972,33.52836114,1,18.2387,17.9746,17.875,17.8532,17.7821,0.004157,...,0.021125,146.514792,33.528250,SN II,0.03800,ZTF18aacemcn,0.597908,0,1,4
1,153731349546771375,134.95471695,38.10897029,1,18.7478,19.0585,18.6466,18.8301,18.6221,0.006392,...,0.01741071,134.954667,38.109056,SN II,0.07247,ZTF18aacnlxz,0.338392,0,1,4
2,169101206490177628,120.64903126,50.9224606,1,16.5325,15.6522,15.2551,15.0064,14.7582,0.002335,...,0.07590476,120.648958,50.922528,SN Ia,0.05295,ZTF18aadlaxo,0.293045,1,0,0
3,159661026075384557,102.60753619,43.05326583,1,nan,14.2309,13.166,12.9816,12.8047,nan,...,0.1546488,102.607542,43.053222,SN IIn,0.01885,ZTF18aadmssd,0.157648,0,1,3
4,146001295340652041,129.53408241,31.66789412,1,17.2694,16.9697,16.8672,16.8232,16.7636,0.004272,...,0.00466667,129.533958,31.667917,SN II,0.03057,ZTF18aadsuxd,0.388737,0,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4896,91140220649260935,22.06491056,-14.04954556,1,18.0565,17.3057,16.8992,16.6476,16.4545,0.005967,...,0.00175,22.065450,-14.051486,SNIa-norm,0.10400,NaN,7.235150,1,0,0
4897,122193229064908516,322.90645396,11.83166024,1,14.7039,13.9191,13.5128,13.2656,13.0388,0.001149,...,0.1600238,322.906997,11.832946,SNIa-norm,0.02500,NaN,5.008624,1,0,0
4898,119713173483364644,317.34832855,9.76180289,1,16.943,16.4661,16.0112,15.9251,15.7482,0.004014,...,0.02083333,317.348935,9.760885,SNIbn,0.08000,NaN,3.943161,0,2,1
4899,158721402117007558,140.21168731,42.27242849,1,16.4714,16.0417,15.8206,15.6633,15.5445,0.00294,...,0.04576786,140.211901,42.272211,SNIa-norm,0.03300,NaN,0.968020,1,0,0


In [26]:
panstarr_tab = pd.read_csv('/Users/adamboesky/Research/ay98/clean_data/panstarrs_hosts_prost/merged.ecsv')
panstarr_tab

,name,RA,DEC,redshift,host_id,host_ra,host_dec,host_prob,host_redshift,host_2_id,...,beta,raStack,yKronRad,cx,yFmeanflxR5,objAltName1,zstackDetectID,yinfoFlag2,skyCellID,raStackErr
0,SN2019vup,08:35:33.780,+43:22:16.85,0.079880,1.600413e+17,128.890402,43.371265,0.999924,0.079613,NaN,...,23.890809,128.890381,5.65851,-0.456384,0.000525,NaN,2.523225e+18,524416.0,81.0,0.001
1,SN2019wmu,06:55:03.351,+54:46:10.61,0.055000,1.737210e+17,103.763532,54.769624,0.966659,0.083766,NaN,...,31.773341,103.763522,4.70010,-0.137245,0.000188,NaN,2.523247e+18,524416.0,66.0,0.001
2,SN2019xbi,10:25:22.211,+11:07:14.48,0.095000,1.213416e+17,156.342459,11.120717,0.939608,0.303589,NaN,...,1.165611,156.342474,1.98601,-0.898761,0.000018,NaN,2.523235e+18,655360.0,78.0,0.001
3,SN2019wiv,09:48:30.414,+57:14:24.17,0.059267,1.766815e+17,147.126643,57.240281,0.999978,0.059577,NaN,...,40.898656,147.126608,7.28601,-0.454469,0.001336,NaN,2.523249e+18,524416.0,30.0,0.001
4,SN2019wku,05:17:21.489,+12:31:02.39,0.036000,1.230208e+17,79.338595,12.517270,0.998055,0.030563,NaN,...,-10.531120,79.338600,9.25961,0.180607,0.000397,NaN,2.523168e+18,128.0,11.0,0.001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8416,SN2020nei,13:07:35.908,+43:33:36.05,0.062000,1.602720e+17,196.900557,43.561219,0.997291,0.061992,NaN,...,45.730112,196.900590,12.28920,-0.693342,0.000725,NaN,2.523030e+18,1572992.0,83.0,0.001
8417,SN2020ppc,20:45:17.061,+07:06:47.83,0.100000,1.165331e+17,311.321016,7.113492,0.530858,0.181464,NaN,...,24.208380,311.321025,NaN,0.655195,0.000050,NaN,2.523021e+18,4.0,75.0,0.001
8418,SN2020oyd,20:04:29.829,+09:20:00.31,0.075000,1.191930e+17,301.125279,9.332236,0.996822,0.068748,NaN,...,28.998318,301.125279,7.85536,0.510070,0.000834,NaN,2.523006e+18,0.0,30.0,0.001
8419,SN2020oco,19:28:09.758,+52:53:38.32,0.026581,1.714729e+17,292.040793,52.894892,0.814694,0.027492,NaN,...,72.583173,292.040798,22.34840,0.226391,0.002111,NaN,2.523009e+18,128.0,28.0,0.001


In [27]:
def propocess_tab(df: pd.DataFrame) -> pd.DataFrame:

    # Make and format a pd dataframe
    column_types = {col: float for col in df.columns[:-2]}
    column_types[df.columns[-2]] = str
    column_types[df.columns[-1]] = float
    df.replace('nan', np.nan, inplace=True)

    # The columns of grizy mags
    sed_cols = ['gKronMag','rKronMag','iKronMag','zKronMag','yKronMag']

    # Drop rows where the specified columns have more than 4 NaN values
    orig_len = len(df)
    df = df.dropna(thresh=3, subset=sed_cols, ignore_index=True)
    # df = df.dropna(subset=['sn_class'], ignore_index=True)
    print(f'Dropped {orig_len - len(df)}/{orig_len} rows due to missingness. Now have {len(df)}')

    # Apply the function to each row
    # df['angular_separation_arcsec'] = df.apply(lambda row: get_angular_separation(row['RA'], row['DEC'], row['host_ra'], row['host_dec'], unit1=(u.hourangle, 'deg'), unit2='deg'), axis=1)

    return df

In [28]:
# Get the preprocessed pstar data
panstarr_df = propocess_tab(panstarr_tab)
panstarr_df

Dropped 1208/8421 rows due to missingness. Now have 7213


,name,RA,DEC,redshift,host_id,host_ra,host_dec,host_prob,host_redshift,host_2_id,...,beta,raStack,yKronRad,cx,yFmeanflxR5,objAltName1,zstackDetectID,yinfoFlag2,skyCellID,raStackErr
0,SN2019vup,08:35:33.780,+43:22:16.85,0.079880,1.600413e+17,128.890402,43.371265,0.999924,0.079613,NaN,...,23.890809,128.890381,5.65851,-0.456384,0.000525,NaN,2.523225e+18,524416.0,81.0,0.001
1,SN2019wmu,06:55:03.351,+54:46:10.61,0.055000,1.737210e+17,103.763532,54.769624,0.966659,0.083766,NaN,...,31.773341,103.763522,4.70010,-0.137245,0.000188,NaN,2.523247e+18,524416.0,66.0,0.001
2,SN2019xbi,10:25:22.211,+11:07:14.48,0.095000,1.213416e+17,156.342459,11.120717,0.939608,0.303589,NaN,...,1.165611,156.342474,1.98601,-0.898761,0.000018,NaN,2.523235e+18,655360.0,78.0,0.001
3,SN2019wiv,09:48:30.414,+57:14:24.17,0.059267,1.766815e+17,147.126643,57.240281,0.999978,0.059577,NaN,...,40.898656,147.126608,7.28601,-0.454469,0.001336,NaN,2.523249e+18,524416.0,30.0,0.001
4,SN2019wku,05:17:21.489,+12:31:02.39,0.036000,1.230208e+17,79.338595,12.517270,0.998055,0.030563,NaN,...,-10.531120,79.338600,9.25961,0.180607,0.000397,NaN,2.523168e+18,128.0,11.0,0.001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7208,SN2020pwn,16:03:04.126,+19:10:56.55,0.046140,1.310124e+17,240.767194,19.182368,0.999998,0.054827,NaN,...,38.999016,240.767253,10.62190,-0.461245,0.002884,NaN,2.522984e+18,524416.0,79.0,0.001
7209,SN2020nei,13:07:35.908,+43:33:36.05,0.062000,1.602720e+17,196.900557,43.561219,0.997291,0.061992,NaN,...,45.730112,196.900590,12.28920,-0.693342,0.000725,NaN,2.523030e+18,1572992.0,83.0,0.001
7210,SN2020ppc,20:45:17.061,+07:06:47.83,0.100000,1.165331e+17,311.321016,7.113492,0.530858,0.181464,NaN,...,24.208380,311.321025,NaN,0.655195,0.000050,NaN,2.523021e+18,4.0,75.0,0.001
7211,SN2020oyd,20:04:29.829,+09:20:00.31,0.075000,1.191930e+17,301.125279,9.332236,0.996822,0.068748,NaN,...,28.998318,301.125279,7.85536,0.510070,0.000834,NaN,2.523006e+18,0.0,30.0,0.001


In [29]:
from astropy.coordinates import SkyCoord, search_around_sky

In [30]:
pstarr_coords = SkyCoord(ra=panstarr_df['host_ra'], dec=panstarr_df['host_dec'], unit='deg')
bts_yse_coords = SkyCoord(ra=sn_df['raStack_3pi'], dec=sn_df['decStack_3pi'], unit='deg')

# Calculate angular separations
pstarr_idxs, bts_yse_idxs, d2d, d3d = search_around_sky(pstarr_coords, bts_yse_coords, 0.01*u.deg)
len(pstarr_idxs), len(pstarr_coords), len(bts_yse_coords)

(2300, 7213, 4901)